<a href="https://colab.research.google.com/github/192565027simats/CSA6102/blob/main/EXP14-Phishing%20Email%20Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re
from urllib.parse import urlparse

# -------------------------------
# Function to Analyze an Email
# -------------------------------
def analyze_email(sender, subject, body):
    report = {}
    indicators = []
    risk_score = 0

    # Sender Verification
    trusted_domains = ["paypal.com", "google.com", "amazon.com", "microsoft.com"]

    sender_domain = sender.split("@")[-1].lower()
    report["Sender"] = sender
    report["Domain"] = sender_domain

    if sender_domain in trusted_domains:
        report["Sender Status"] = "Legitimate"
    else:
        report["Sender Status"] = "Suspicious"
        indicators.append("Unknown or spoofed sender domain")
        risk_score += 20

    # ---------------------------------
    # Extract URLs
    # ---------------------------------
    urls = re.findall(r'https?://[^\s]+', body)

    report["URLs"] = urls

    suspicious_urls = []

    for url in urls:
        parsed = urlparse(url)
        hostname = parsed.hostname if parsed.hostname else ""

        # Check if hostname is an IP address
        if re.fullmatch(r'(\d{1,3}\.){3}\d{1,3}', hostname):
            suspicious_urls.append(url)
            indicators.append("Raw IP address used in URL")
            risk_score += 25

        # Suspicious keywords
        if any(word in url.lower() for word in ["verify", "login", "secure", "update"]):
            suspicious_urls.append(url)
            indicators.append("Suspicious URL keywords")
            risk_score += 15

    # ---------------------------------
    # Phishing Keywords
    # ---------------------------------
    phishing_keywords = [
        "urgent",
        "verify your account",
        "click here",
        "account suspended",
        "password",
        "login",
        "update"
    ]

    for keyword in phishing_keywords:
        if keyword.lower() in body.lower():
            indicators.append(f'Keyword detected: "{keyword}"')
            risk_score += 5

    # ---------------------------------
    # Risk Classification
    # ---------------------------------
    if risk_score > 100:
        risk_score = 100

    if risk_score >= 70:
        verdict = "HIGH RISK (Phishing)"
    elif risk_score >= 40:
        verdict = "MEDIUM RISK"
    else:
        verdict = "LOW RISK / Legitimate"

    report["Suspicious URLs"] = suspicious_urls
    report["Indicators"] = indicators
    report["Risk Score"] = risk_score
    report["Verdict"] = verdict

    return report


# ---------------------------------
# Sample Emails
# ---------------------------------
emails = [
    {
        "sender": "support@paypal.com",
        "subject": "Monthly Statement",
        "body": "Please find your monthly statement attached."
    },
    {
        "sender": "alert@paypal-secure-verify.com",
        "subject": "URGENT: Verify Your Account",
        "body": """
Your account has been suspended.

Click here immediately:

http://192.168.10.5/login

Verify your account within 24 hours.
"""
    }
]

# ---------------------------------
# Analyze Emails
# ---------------------------------
phishing_count = 0
legitimate_count = 0
total_urls = 0
suspicious_urls = 0

print("=" * 60)
print("PHISHING EMAIL ANALYSIS REPORT")
print("=" * 60)

for i, email in enumerate(emails, start=1):

    result = analyze_email(
        email["sender"],
        email["subject"],
        email["body"]
    )

    print(f"\nEmail {i}")
    print("-" * 50)
    print("Sender          :", result["Sender"])
    print("Sender Status   :", result["Sender Status"])
    print("Domain          :", result["Domain"])

    print("\nURLs Found:")
    if result["URLs"]:
        for url in result["URLs"]:
            print(" -", url)
    else:
        print(" None")

    print("\nIndicators of Compromise:")
    if result["Indicators"]:
        for item in result["Indicators"]:
            print(" -", item)
    else:
        print(" None")

    print("\nRisk Score :", result["Risk Score"])
    print("Verdict    :", result["Verdict"])

    total_urls += len(result["URLs"])
    suspicious_urls += len(result["Suspicious URLs"])

    if "Phishing" in result["Verdict"]:
        phishing_count += 1
    else:
        legitimate_count += 1

# ---------------------------------
# Summary Statistics
# ---------------------------------
print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print("Total Emails Analyzed :", len(emails))
print("Legitimate Emails     :", legitimate_count)
print("Phishing Emails       :", phishing_count)
print("URLs Extracted        :", total_urls)
print("Suspicious URLs       :", suspicious_urls)

PHISHING EMAIL ANALYSIS REPORT

Email 1
--------------------------------------------------
Sender          : support@paypal.com
Sender Status   : Legitimate
Domain          : paypal.com

URLs Found:
 None

Indicators of Compromise:
 None

Risk Score : 0
Verdict    : LOW RISK / Legitimate

Email 2
--------------------------------------------------
Sender          : alert@paypal-secure-verify.com
Sender Status   : Suspicious
Domain          : paypal-secure-verify.com

URLs Found:
 - http://192.168.10.5/login

Indicators of Compromise:
 - Unknown or spoofed sender domain
 - Raw IP address used in URL
 - Suspicious URL keywords
 - Keyword detected: "verify your account"
 - Keyword detected: "click here"
 - Keyword detected: "login"

Risk Score : 75
Verdict    : HIGH RISK (Phishing)

SUMMARY STATISTICS
Total Emails Analyzed : 2
Legitimate Emails     : 1
Phishing Emails       : 1
URLs Extracted        : 1
Suspicious URLs       : 2
